<a href="https://colab.research.google.com/github/peemoshi/ai-ml-internship/blob/main/week-1/1.4-scikit-learn-models/sklearn_models_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/student_mat_cleaned.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


This notebook builds and evaluates two machine learning models: a regression model predicting a student's final grade, and a classification model predicting pass/fail, using the same UCI Student Performance dataset cleaned in Task 1.2.

Students who did not complete the final exam (G3 = 0) are excluded from modeling, since predicting a withdrawal is a different problem from predicting academic performance.

The prior-term grades G1 and G2 are also excluded as features, since they are strongly correlated with G3 and would make the prediction trivial; this way, the model has to learn from genuine demographic and behavioral factors instead. Numeric features are scaled and categorical features are one-hot encoded using a ColumnTransformer, so both types can be fed into a model consistently.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.read_csv("/content/drive/MyDrive/student_mat_cleaned.csv")   # your cleaned file from Task 1.2
df_model = df[df['completed_exam'] == True].copy()
print("Rows used for modeling:", len(df_model))

target_reg = 'G3'
drop_cols = ['G1', 'G2', 'G3', 'completed_exam']
X = df_model.drop(columns=drop_cols)
y_reg = df_model[target_reg]

numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()
print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

Rows used for modeling: 357
Numeric: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']
Categorical: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


This cell builds the regression model, which predicts a student's final grade (G3) as a number. Before training a real model, a baseline is trained that simply predicts the average grade for every student, this gives a reference point to check whether the real model is actually learning something useful.

A Linear Regression model is then trained on the same data and compared against the baseline using three metrics: MAE (average size of the error), RMSE (similar to MAE but penalizes large errors more), and R² (the proportion of variation in grades the model explains).

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)

# Baseline: always predict the average grade (gives us a reference point)
baseline = Pipeline([('preprocessor', preprocessor), ('model', DummyRegressor(strategy='mean'))])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

# Real model
reg_model = Pipeline([('preprocessor', preprocessor), ('model', LinearRegression())])
reg_model.fit(X_train, y_train)
reg_pred = reg_model.predict(X_test)

def report_regression(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}")

report_regression("Baseline (mean)", y_test, baseline_pred)
report_regression("Linear Regression", y_test, reg_pred)

Baseline (mean): MAE=2.57  RMSE=3.11  R²=-0.001
Linear Regression: MAE=2.32  RMSE=2.77  R²=0.202


This cell builds the classification model, which predicts whether a student passes or fails (G3 >= 10, Portugal's passing mark out of 20). As with regression, a baseline is trained first — one that always predicts the majority class ("pass") — followed by a real Logistic Regression model. Because significantly more students pass than fail in this dataset, accuracy alone can be misleading, so precision, recall, F1, and a confusion matrix are all reported to get a fuller picture of how well each model actually identifies at-risk (failing) students.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_class = (df_model['G3'] >= 10).astype(int)   # 1 = pass, 0 = fail (10/20 is the passing mark in Portugal)
print("Pass:", y_class.sum(), " Fail:", (y_class == 0).sum())

Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, y_class, test_size=0.2, random_state=42, stratify=y_class)

baseline_clf = Pipeline([('preprocessor', preprocessor), ('model', DummyClassifier(strategy='most_frequent'))])
baseline_clf.fit(Xc_train, yc_train)
baseline_clf_pred = baseline_clf.predict(Xc_test)

clf_model = Pipeline([('preprocessor', preprocessor), ('model', LogisticRegression(max_iter=1000))])
clf_model.fit(Xc_train, yc_train)
clf_pred = clf_model.predict(Xc_test)

def report_classification(name, y_true, y_pred):
    print(f"\n{name}")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"Recall: {recall_score(y_true, y_pred):.3f}")
    print(f"F1: {f1_score(y_true, y_pred):.3f}")
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

report_classification("Baseline (most frequent)", yc_test, baseline_clf_pred)
report_classification("Logistic Regression", yc_test, clf_pred)

Pass: 265  Fail: 92

Baseline (most frequent)
Accuracy: 0.736
Precision: 0.736
Recall: 1.000
F1: 0.848
Confusion matrix:
 [[ 0 19]
 [ 0 53]]

Logistic Regression
Accuracy: 0.681
Precision: 0.768
Recall: 0.811
F1: 0.789
Confusion matrix:
 [[ 6 13]
 [10 43]]


Beyond just reporting the metrics, this cell looks directly at students the Logistic Regression model got wrong, to understand what it struggled with rather than just how often it was wrong. Inspecting these specific cases (rather than only aggregate scores) helps identify whether errors follow a pattern; for example, if the model consistently misjudges students with certain combinations of study habits or prior failures, which points toward realistic next steps for improving the model.

In [7]:
# Look at a few misclassified students
results = Xc_test.copy()
results['actual'] = yc_test.values
results['predicted'] = clf_pred
misclassified = results[results['actual'] != results['predicted']]
print(f"{len(misclassified)} misclassified out of {len(results)}")
misclassified[['failures', 'studytime', 'absences', 'higher', 'actual', 'predicted']].head(10)

23 misclassified out of 72


,failures,studytime,absences,higher,actual,predicted
314,2,3,14,yes,1,0
201,0,2,6,yes,1,0
1,0,2,4,yes,0,1
40,1,2,25,yes,1,0
2,3,2,10,yes,1,0
175,0,2,4,yes,0,1
67,0,4,4,yes,0,1
89,0,2,18,yes,0,1
280,0,1,30,yes,0,1
304,1,2,20,yes,1,0


**Summary:**

Two models were built on the same student dataset: a Linear Regression model predicting final grade (R² = 0.20) and a Logistic Regression model predicting pass/fail (F1 = 0.79). Both beat their respective baselines, but using only demographic and behavioral features (deliberately excluding prior grades) means a large share of the variation in outcomes remains unexplained, likely driven by factors not captured in this dataset, like individual ability or teacher effects. The classification results also showed why accuracy alone can be misleading on imbalanced data: a baseline with no real skill scored higher on accuracy but caught zero failing students, while the real model traded some accuracy for actually identifying at-risk students. Limitations: small dataset (357 usable rows), no hyperparameter tuning, and no cross-validation.